# ⚡ RFB-ESRGAN - Google Colab Pro Optimized Configuration

## Colab Pro Specifications:
- **GPU**: NVIDIA Tesla T4/P100/V100 (depending on availability)
- **GPU Memory**: 15-16 GB
- **RAM**: 25-52 GB
- **Session Limit**: 24 hours continuous

## Performance Improvements Applied:
- **HR Image Size**: 256 pixels (optimized for memory)
- **Batch Size**: 16 (maximizes Colab Pro GPU utilization)
- **Stage 1 Epochs**: 10 (faster initial training)
- **Stage 2 Iterations**: 50,000 (balanced quality/time)
- **Google Drive Integration**: Persistent storage across sessions

## Expected Training Time:
- **Stage 1**: ~1-2 hours
- **Stage 2**: ~3-4 hours
- **Total**: ~4-6 hours ⚡

## Quality Trade-offs:
- Output resolution: 256x256 (8x upscaling from 32x32)
- Perceptual quality maintained through GAN training
- Ensemble of 10 models for best results

## Important Notes:
- Colab sessions timeout after 12 hours of inactivity
- Models are automatically saved to Google Drive
- You can resume training by loading saved checkpoints

---

In [ ]:
# Cell 1.5: Install and Setup Kaggle Dataset Download

# Install kagglehub if not already installed
!pip install -q kagglehub

import kagglehub
import pandas as pd
import json

# Download the label-indices dataset
print("📦 Downloading dataset from Kaggle...")
dataset_path = kagglehub.dataset_download("supernovahegde/label-indices")
print(f"✓ Dataset downloaded to: {dataset_path}")

# Load the CSV files
train_df = pd.read_csv(f"{dataset_path}/train.csv")
val_df = pd.read_csv(f"{dataset_path}/val.csv")
test_df = pd.read_csv(f"{dataset_path}/test.csv")

# Load label indices
with open(f"{dataset_path}/label_indices.json", 'r') as f:
    label_indices = json.load(f)

print(f"\n📊 Dataset Statistics:")
print(f"  • Training samples: {len(train_df)}")
print(f"  • Validation samples: {len(val_df)}")
print(f"  • Test samples: {len(test_df)}")
print(f"  • Number of classes: {len(label_indices)}")
print(f"\n🏷️ Label Indices: {label_indices}")

# Display sample data
print(f"\n📋 Training Data Sample:")
print(train_df.head())
print(f"\n📋 Training Data Columns:")
print(train_df.columns.tolist())

# Store dataset path for later use
DATASET_PATH = dataset_path

# RFB-ESRGAN for BigEarthNet Super-Resolution - Google Colab Pro

## Setup Instructions for Google Colab

**IMPORTANT:** Follow these steps in order:

1. **Run Cell 1 ONCE** to:
   - Mount Google Drive
   - Install all dependencies
   - Set up WandB logging

2. **Upload Dataset to Google Drive:**
   - Upload your BigEarthNet-S2 dataset to: `/content/drive/MyDrive/BigEarthNet-S2/`
   - Or modify the `BIGEARTHNET_DIR` path in Cell 4 to point to your dataset location

3. **Restart Runtime** (Optional, only if you see CUDA version mismatch):
   - Go to: Runtime → Restart runtime
   - Skip Cell 1 and run from Cell 2 onwards

4. **Run remaining cells** in sequence

## Colab Pro Optimizations Applied:

- ✅ Google Drive integration for dataset and model storage
- ✅ Batch size optimized for Colab Pro GPU (16)
- ✅ Memory management with `torch.cuda.empty_cache()`
- ✅ Reduced num_workers (2) to avoid multiprocessing issues
- ✅ All outputs saved to Google Drive for persistence
- ✅ Single GPU configuration (standard for Colab)

## Output Location:
All trained models will be saved to:
`/content/drive/MyDrive/RFB-ESRGAN-Output/`

---

In [ ]:
# Cell 1: Imports and Setup

# Mount Google Drive first
from google.colab import drive
drive.mount('/content/drive')

# Install required dependencies
# Uninstall old versions and install compatible PyTorch/torchvision
!pip uninstall -y torch torchvision torchaudio
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q wandb tqdm pillow numpy opencv-python scikit-image rasterio

print("✓ All dependencies installed successfully!")
print("⚠️ If you see CUDA version mismatch errors, restart the runtime and run from Cell 2")

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import glob
from tqdm import tqdm
import wandb
from collections import OrderedDict
import time

# WandB setup - You can change this to your own key
wandb.login(key='5424a3d65aac1662f5be82d4439aaac35046689e')
wandb.init(
    project='RFB-ESRGAN-BigEarthNet-Colab',
    config={
        'upscale_factor': 8,  # 32→256
        'lr_size': 32,
        'hr_size': 256,
        'batch_size': 16,  # Colab Pro can handle batch size 16
        'stage1_epochs': 20,  # Reduced from 50 to 20 to prevent discriminator collapse
        'stage2_iters': 200000,  # Increased from 50k to 200k (~14-15 hours)
        'stage2_warmup_iters': 5000,  # Discriminator warmup iterations
        'stage1_lr': 1e-4,
        'stage2_lr': 5e-5,
        'lambda_pix': 1.0,
        'lambda_vgg': 1.0,
        'lambda_adv': 5e-3,  # Increased from 1e-3 to 5e-3 for stronger adversarial signal
        'num_rrdb': 12,
        'num_rrfdb': 6,
        'ensemble_models': 10,
        'grad_clip': 0.1,
        'd_updates_per_g': 3,  # Train discriminator 3x per generator update
        'target_runtime_hours': 18  # Target 18 hours total runtime
    }
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')
print(f'Torchvision version: {torchvision.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version: {torch.version.cuda}')
    print(f'Number of GPUs available: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
    # Print GPU memory info
print(f'\n⚡ COLAB PRO OPTIMIZED CONFIGURATION (18-HOUR RUNTIME - FIXED DISCRIMINATOR):')

print(f'  • Stage 1: 20 epochs (~1-2 hours, prevents generator from getting too strong)')
print(f'  • Stage 2: 200,000 iterations (~16-17 hours)')
print(f'  • Discriminator warmup: 5,000 iterations')
print(f'  • Total estimated runtime: ~18 hours')
print(f'  • Learning rate: 1e-4 (stable)')
print(f'  • Lambda_adv: 5e-3 (5x stronger adversarial signal)')
print(f'  • Discriminator updates: 3x per generator update')
print(f'  • Model: 12 RRDBs, 6 RRFDBs with Spectral Normalization')
print(f'  • Gradient clipping: 0.1 (prevents explosion)')
print(f'  • Google Drive mounted for data storage')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Create output directory in Google Drive

os.makedirs('/content/drive/MyDrive/RFB-ESRGAN-Output', exist_ok=True)
os.makedirs('/content/drive/MyDrive/RFB-ESRGAN-Output', exist_ok=True)print(f'\n✓ Output directory created: /content/drive/MyDrive/RFB-ESRGAN-Output')
print(f'\n✓ Output directory created: /content/drive/MyDrive/RFB-ESRGAN-Output')

## 📊 Extended Training Configuration - 18 Hour Runtime (FIXED DISCRIMINATOR)

This configuration is optimized for a **~18 hour training session** on Google Colab Pro with **discriminator collapse fixes**:

### ✅ Discriminator Collapse Fixes Applied:
1. **Reduced Stage 1**: 20 epochs (not 50) - prevents generator from getting too strong
2. **Stronger adversarial signal**: λ_adv = 5e-3 (5x increase from 1e-3)
3. **Multiple discriminator updates**: 3 updates per generator update
4. **Spectral Normalization**: Replaced BatchNorm in discriminator
5. **Gradient clipping**: Applied to both G and D (0.1)
6. **Discriminator warmup**: 5,000 iterations of D-only training

### Training Schedule:
- **Stage 1 (PSNR-oriented)**: 20 epochs (~1-2 hours)
  - Pure L1 pixel loss training
  - Shorter to prevent generator dominance
  - Learning rate: 1e-4 with StepLR decay

- **Discriminator Warmup**: 5,000 iterations (~15 minutes)
  - Train discriminator alone to catch up with generator
  - Helps prevent immediate collapse

- **Stage 2 (GAN training)**: 200,000 iterations (~16-17 hours)
  - Combined perceptual + adversarial loss
  - 3:1 discriminator-to-generator update ratio
  - Stronger adversarial weight (5e-3 vs 1e-3)
  - Learning rate milestones at: 50k, 100k, 150k, 180k
  - Checkpoints saved every 10k iterations

### Performance Estimates:
- **Total runtime**: ~18 hours
- **Checkpoints saved**: 20 models (every 10k iterations)
- **Ensemble**: Top 10 models averaged for final model
- **Expected improvement**: Sharp, detailed GAN-quality results (not smooth/blurry)

### Why These Fixes?
- **Balanced training**: Generator and discriminator start at similar strength levels
- **Stability**: Spectral norm + gradient clipping prevent training instabilities
- **Stronger signal**: Higher λ_adv means generator actually listens to discriminator
- **Better convergence**: Multiple D updates keep discriminator competitive

In [ ]:
# Cell 2: Model Architecture - RFB, RRDB, RRFDB Blocks

class RFB(nn.Module):
    """Receptive Field Block - Multi-scale feature extraction with small kernels"""
    def __init__(self, in_channels=64):
        super(RFB, self).__init__()
        # Branch 1: AvgPool(3) + 1x1 conv + dilated 3x3 (d=1)
        self.branch1 = nn.Sequential(
            nn.AvgPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, 16, 1, 1, 0),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 16, 3, 1, padding=1, dilation=1),
            nn.ReLU(inplace=True)
        )
        
        # Branch 2: AvgPool(5) + 1x1 conv + dilated 3x3 (d=2)
        self.branch2 = nn.Sequential(
            nn.AvgPool2d(5, stride=1, padding=2),
            nn.Conv2d(in_channels, 24, 1, 1, 0),
            nn.ReLU(inplace=True),
            nn.Conv2d(24, 24, 3, 1, padding=2, dilation=2),
            nn.ReLU(inplace=True)
        )
        
        # Branch 3: AvgPool(7) + 1x1 conv + dilated 3x3 (d=3)
        self.branch3 = nn.Sequential(
            nn.AvgPool2d(7, stride=1, padding=3),
            nn.Conv2d(in_channels, 24, 1, 1, 0),
            nn.ReLU(inplace=True),
            nn.Conv2d(24, 24, 3, 1, padding=3, dilation=3),
            nn.ReLU(inplace=True)
        )
        
        # Concat 16+24+24=64 → 1x1 conv to 64
        self.conv_concat = nn.Sequential(
            nn.Conv2d(64, in_channels, 1, 1, 0),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        concat = torch.cat([b1, b2, b3], dim=1)
        out = self.conv_concat(concat)
        return out


class DenseBlock(nn.Module):
    """Dense Block with 5 convolutions (from ESRGAN RRDB)"""
    def __init__(self, nf=64, gc=32):
        super(DenseBlock, self).__init__()
        self.conv1 = nn.Conv2d(nf, gc, 3, 1, 1)
        self.conv2 = nn.Conv2d(nf + gc, gc, 3, 1, 1)
        self.conv3 = nn.Conv2d(nf + 2 * gc, gc, 3, 1, 1)
        self.conv4 = nn.Conv2d(nf + 3 * gc, gc, 3, 1, 1)
        self.conv5 = nn.Conv2d(nf + 4 * gc, nf, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat([x, x1], dim=1)))
        x3 = self.lrelu(self.conv3(torch.cat([x, x1, x2], dim=1)))
        x4 = self.lrelu(self.conv4(torch.cat([x, x1, x2, x3], dim=1)))
        x5 = self.conv5(torch.cat([x, x1, x2, x3, x4], dim=1))
        return x5 * 0.2 + x  # Residual scaling


class RRDB(nn.Module):
    """Residual-in-Residual Dense Block (ESRGAN)"""
    def __init__(self, nf=64, gc=32):
        super(RRDB, self).__init__()
        self.db1 = DenseBlock(nf, gc)
        self.db2 = DenseBlock(nf, gc)
        self.db3 = DenseBlock(nf, gc)

    def forward(self, x):
        out = self.db1(x)
        out = self.db2(out)
        out = self.db3(out)
        return out * 0.2 + x  # Residual scaling


class RRFDB(nn.Module):
    """Residual Receptive Field Dense Block (5 RFBs in dense style)"""
    def __init__(self, nf=64):
        super(RRFDB, self).__init__()
        self.rfb1 = RFB(nf)
        self.rfb2 = RFB(nf)
        self.rfb3 = RFB(nf)
        self.rfb4 = RFB(nf)
        self.rfb5 = RFB(nf)
        # Simple dense connection via addition (simplified from paper)

    def forward(self, x):
        out = self.rfb1(x)
        out = self.rfb2(out)
        out = self.rfb3(out)
        out = self.rfb4(out)
        out = self.rfb5(out)
        return out * 0.2 + x  # Residual scaling


class Generator(nn.Module):
    """RFB-ESRGAN Generator - x8 upscale (32→256) optimized version"""
    def __init__(self, num_rrdb=16, num_rrfdb=8, nf=64):
        super(Generator, self).__init__()
        # First conv
        self.conv_first = nn.Conv2d(3, nf, 3, 1, 1)
        
        # Trunk-A: 16 RRDBs
        self.trunk_a = nn.Sequential(*[RRDB(nf) for _ in range(num_rrdb)])
        
        # Trunk-RFB: 8 RRFDBs
        self.trunk_rfb = nn.Sequential(*[RRFDB(nf) for _ in range(num_rrfdb)])
        
        # Single RFB before upsampling
        self.rfb_up = RFB(nf)
        
        # Upsampling for x8 total: x2 → x2 → x2 (32 → 64 → 128 → 256)
        # Changed from x16 to x8 to match hr_size=256
        self.upsample = nn.Sequential(
            nn.Conv2d(nf, nf * 4, 3, 1, 1),
            nn.PixelShuffle(2),  # x2: 32 → 64
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(nf, nf * 4, 3, 1, 1),
            nn.PixelShuffle(2),  # x2: 64 → 128
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(nf, nf * 4, 3, 1, 1),
            nn.PixelShuffle(2),  # x2: 128 → 256
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # Final convs
        self.conv_final = nn.Sequential(
            nn.Conv2d(nf, nf, 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(nf, 3, 3, 1, 1),
            nn.Tanh()
        )
        
    def forward(self, x):
        feat = self.conv_first(x)
        trunk_a_out = self.trunk_a(feat)
        trunk_rfb_out = self.trunk_rfb(trunk_a_out)
        rfb_out = self.rfb_up(trunk_rfb_out)
        upsampled = self.upsample(rfb_out)
        out = self.conv_final(upsampled)
        return out


class Discriminator(nn.Module):
    """ESRGAN-style Discriminator with Spectral Normalization (no BatchNorm)"""
    def __init__(self, in_channels=3, nf=64):
        super(Discriminator, self).__init__()
        
        def conv_block(in_c, out_c, stride=1, use_sn=True):
            layers = []
            conv = nn.Conv2d(in_c, out_c, 3, stride, 1)
            if use_sn:
                # Apply Spectral Normalization for stability
                conv = nn.utils.spectral_norm(conv)
            layers.append(conv)
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)
        
        self.features = nn.Sequential(
            conv_block(in_channels, nf, 1, False),  # First layer no SN
            conv_block(nf, nf, 2),
            conv_block(nf, nf * 2, 1),
            conv_block(nf * 2, nf * 2, 2),
            conv_block(nf * 2, nf * 4, 1),
            conv_block(nf * 4, nf * 4, 2),
            conv_block(nf * 4, nf * 8, 1),
            conv_block(nf * 8, nf * 8, 2),
        )
        
        # Spectral norm on final linear layer too
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.utils.spectral_norm(nn.Linear(nf * 8, 1))
        )
        
    def forward(self, x):
        feat = self.features(x)
        out = self.classifier(feat)
        return out

print("  • Using PixelShuffle for efficient upsampling")

print("✓ Architecture defined successfully!")print("  • Generator: x8 upscale (32→256)")

Architecture defined successfully!


In [3]:
# Cell 3: Loss Functions

class VGGPerceptualLoss(nn.Module):
    """VGG19 conv3_4 perceptual loss (L_VGG)"""
    def __init__(self):
        super(VGGPerceptualLoss, self).__init__()
        vgg = torchvision.models.vgg19(pretrained=True).features
        self.vgg_layers = nn.Sequential(*list(vgg.children())[:16])  # Up to conv3_4
        for param in self.vgg_layers.parameters():
            param.requires_grad = False
        self.vgg_layers.eval()
        
        # ImageNet normalization
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        
    def forward(self, sr, hr):
        # Normalize from [-1,1] to ImageNet range
        sr = (sr + 1) / 2  # [0,1]
        hr = (hr + 1) / 2
        sr = (sr - self.mean) / self.std
        hr = (hr - self.mean) / self.std
        
        sr_feat = self.vgg_layers(sr)
        hr_feat = self.vgg_layers(hr)
        return F.l1_loss(sr_feat, hr_feat)


class GANLoss(nn.Module):
    """Relativistic GAN loss from ESRGAN"""
    def __init__(self):
        super(GANLoss, self).__init__()
        
    def forward(self, d_real, d_fake, is_disc=False):
        if is_disc:
            # Discriminator loss: L_D = -E[log(Delta_Real)] - E[1-log(Delta_Fake)]
            delta_real = torch.sigmoid(d_real - d_fake.mean())
            delta_fake = torch.sigmoid(d_fake - d_real.mean())
            loss_real = -torch.log(delta_real + 1e-8).mean()
            loss_fake = -torch.log(1 - delta_fake + 1e-8).mean()
            return loss_real + loss_fake
        else:
            # Generator adversarial loss: L_adv = -E[log(1-Delta_Real)] - E[log(Delta_Fake)]
            delta_real = torch.sigmoid(d_real - d_fake.mean())
            delta_fake = torch.sigmoid(d_fake - d_real.mean())
            loss = -torch.log(1 - delta_real + 1e-8).mean() - torch.log(delta_fake + 1e-8).mean()
            return loss


def compute_generator_loss(sr, hr, d_real, d_fake, vgg_loss_fn, gan_loss_fn, lambda_pix=10, lambda_vgg=1, lambda_adv=5e-3):
    """Total generator loss: L_G = λ*L_pix + L_VGG + η*L_adv"""
    l_pix = F.l1_loss(sr, hr)
    l_vgg = vgg_loss_fn(sr, hr)
    l_adv = gan_loss_fn(d_real, d_fake, is_disc=False)
    
    total_loss = lambda_pix * l_pix + lambda_vgg * l_vgg + lambda_adv * l_adv
    
    return total_loss, l_pix, l_vgg, l_adv

print("Loss functions defined successfully!")

Loss functions defined successfully!


In [ ]:
# Cell 3.5: CSV-Based Dataset Loader

class CSVImageDataset(Dataset):
    """
    Dataset class that loads images based on CSV files
    Assumes CSV has columns: 'image_path' (or 'filename') and 'label'
    """
    def __init__(self, csv_path, dataset_path, label_indices, hr_size=256, lr_size=32, mode='train'):
        self.csv_path = csv_path
        self.dataset_path = dataset_path
        self.hr_size = hr_size
        self.lr_size = lr_size
        self.mode = mode
        self.label_indices = label_indices
        
        # Load CSV
        self.df = pd.read_csv(csv_path)
        print(f"📋 Loaded {mode} CSV with {len(self.df)} samples")
        print(f"   Columns: {self.df.columns.tolist()}")
        
        # Detect image path column (common names)
        image_col_names = ['image_path', 'filename', 'file_path', 'path', 'image']
        self.image_col = None
        for col in image_col_names:
            if col in self.df.columns:
                self.image_col = col
                break
        
        if self.image_col is None:
            # Use first column as default
            self.image_col = self.df.columns[0]
            print(f"   ⚠️ Using first column '{self.image_col}' as image path")
        
        # Detect label column
        label_col_names = ['label', 'class', 'target', 'category']
        self.label_col = None
        for col in label_col_names:
            if col in self.df.columns:
                self.label_col = col
                break
        
        if self.label_col:
            print(f"   ✓ Using '{self.image_col}' for images and '{self.label_col}' for labels")
        else:
            print(f"   ✓ Using '{self.image_col}' for images (no labels detected)")
        
        # Transforms for HR images
        if mode == 'train':
            self.hr_transform = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.RandomRotation(degrees=10),
                transforms.Resize((hr_size, hr_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # [-1, 1]
            ])
        else:
            self.hr_transform = transforms.Compose([
                transforms.Resize((hr_size, hr_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
        
        # LR transform (downscale)
        self.lr_transform = transforms.Compose([
            transforms.Resize((lr_size, lr_size), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Get image path
        img_name = self.df.iloc[idx][self.image_col]
        
        # Handle different path formats
        if os.path.isabs(img_name):
            img_path = img_name
        else:
            img_path = os.path.join(self.dataset_path, img_name)
        
        # Try loading the image
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"⚠️ Error loading {img_path}: {e}")
            # Return a black image as fallback
            img = Image.new('RGB', (self.hr_size, self.hr_size), (0, 0, 0))
        
        # Generate HR and LR versions
        hr_img = self.hr_transform(img)
        lr_img = self.lr_transform(img)
        
        # Get label if available
        label = -1  # Default no label
        if self.label_col and self.label_col in self.df.columns:
            label_name = self.df.iloc[idx][self.label_col]
            if label_name in self.label_indices:
                label = self.label_indices[label_name]
            else:
                label = int(label_name) if str(label_name).isdigit() else -1
        
        return lr_img, hr_img, label


# Create dataset instances
print("\n🔄 Creating datasets from CSV files...")
train_dataset = CSVImageDataset(
    csv_path=f"{DATASET_PATH}/train.csv",
    dataset_path=DATASET_PATH,
    label_indices=label_indices,
    hr_size=256,
    lr_size=32,
    mode='train'
)

val_dataset = CSVImageDataset(
    csv_path=f"{DATASET_PATH}/val.csv",
    dataset_path=DATASET_PATH,
    label_indices=label_indices,
    hr_size=256,
    lr_size=32,
    mode='val'
)

test_dataset = CSVImageDataset(
    csv_path=f"{DATASET_PATH}/test.csv",
    dataset_path=DATASET_PATH,
    label_indices=label_indices,
    hr_size=256,
    lr_size=32,
    mode='test'
)

print(f"\n✓ Datasets created successfully!")
print(f"  • Training: {len(train_dataset)} samples")
print(f"  • Validation: {len(val_dataset)} samples")
print(f"  • Test: {len(test_dataset)} samples")

In [ ]:
# Cell 3.6: Create DataLoaders from CSV-based Datasets

from torch.utils.data import DataLoader

# Get batch size from wandb config
batch_size = wandb.config.batch_size

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\n✓ DataLoaders created successfully!")
print(f"  • Training batches: {len(train_loader)} (batch_size={batch_size})")
print(f"  • Validation batches: {len(val_loader)}")
print(f"  • Test batches: {len(test_loader)}")

# Test loading one batch
print(f"\n🧪 Testing data loading...")
try:
    lr_batch, hr_batch, label_batch = next(iter(train_loader))
    print(f"  ✓ LR batch shape: {lr_batch.shape}")
    print(f"  ✓ HR batch shape: {hr_batch.shape}")
    print(f"  ✓ Label batch shape: {label_batch.shape}")
    print(f"  ✓ LR range: [{lr_batch.min():.3f}, {lr_batch.max():.3f}]")
    print(f"  ✓ HR range: [{hr_batch.min():.3f}, {hr_batch.max():.3f}]")
    print(f"  ✓ Sample labels: {label_batch[:5].tolist()}")
except Exception as e:
    print(f"  ⚠️ Error loading batch: {e}")
    print(f"  This might be normal if image paths need adjustment")

In [ ]:
# Cell 4: Dataset and DataLoader for BigEarthNet

import rasterio
from rasterio.errors import RasterioIOError

class BigEarthNetDataset(Dataset):
    """BigEarthNet dataset for super-resolution - handles multi-band Sentinel-2 imagery"""
    def __init__(self, data_dir, hr_size=256, lr_size=32, transform=None, use_rgb_only=True):
        self.data_dir = data_dir
        self.hr_size = hr_size
        self.lr_size = lr_size
        self.use_rgb_only = use_rgb_only
        
        # Find all patch directories (each patch has multiple band files)
        all_paths = glob.glob(os.path.join(data_dir, '**'), recursive=True)
        # Filter to get only directories that contain band files
        self.patch_dirs = []
        for path in all_paths:
            if os.path.isdir(path) and any(f.endswith('.tif') for f in os.listdir(path)):
                self.patch_dirs.append(path)
        
        print(f"Found {len(self.patch_dirs)} image patches in {data_dir}")
        
        # RGB bands for Sentinel-2: B04 (Red), B03 (Green), B02 (Blue)
        self.rgb_bands = ['B04', 'B03', 'B02']  # Order: R, G, B
        
        # Track failed patches (minimal logging to avoid console spam)
        self.failed_count = 0
        
        # Augmentation - Random flips and 90-degree rotations
        self.transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomChoice([
                transforms.RandomRotation([0, 0]),
                transforms.RandomRotation([90, 90]),
                transforms.RandomRotation([180, 180]),
                transforms.RandomRotation([270, 270]),
            ])
        ]) if transform is None else transform
        
        self.to_tensor = transforms.ToTensor()
        
    def __len__(self):
        return len(self.patch_dirs)
    
    def load_rgb_from_patch(self, patch_dir):
        """Load RGB bands from BigEarthNet patch directory"""
        try:
            rgb_arrays = []
            for band in self.rgb_bands:
                # Find the band file in the patch directory
                band_files = glob.glob(os.path.join(patch_dir, f'*_{band}.tif'))
                if not band_files:
                    return None
                
                # Read the band with rasterio - handle numpy type issues
                try:
                    with rasterio.open(band_files[0]) as src:
                        band_data = src.read(1)  # Read first channel
                        # Force conversion to standard numpy array to avoid type conflicts
                        band_data = np.asarray(band_data, dtype=np.float32)
                        # Normalize to 0-255 range (Sentinel-2 data is typically 0-10000)
                        band_data = np.clip(band_data / 10000.0 * 255, 0, 255).astype(np.uint8)
                        rgb_arrays.append(band_data)
                except (TypeError, AttributeError, ValueError) as type_err:
                    # Handle numpy type mismatch errors from rasterio
                    # Skip this patch and return None to use fallback
                    return None
            
            if len(rgb_arrays) != 3:
                return None
                
            # Stack bands into RGB image (Height x Width x 3)
            rgb_image = np.stack(rgb_arrays, axis=-1)
            return Image.fromarray(rgb_image)  # PIL auto-detects RGB mode
            
        except Exception:
            # Silently skip problematic patches to avoid flooding console
            return None
    
    def __getitem__(self, idx):
        patch_dir = self.patch_dirs[idx]
        
        try:
            # Load RGB image from multi-band TIFF files
            img = self.load_rgb_from_patch(patch_dir)
            
            if img is None:
                # Track failures but don't spam console
                self.failed_count += 1
                # Return random tensors as fallback
                return torch.randn(3, self.lr_size, self.lr_size), torch.randn(3, self.hr_size, self.hr_size)
            
            # Random crop to hr_size
            w, h = img.size
            if w < self.hr_size or h < self.hr_size:
                # Resize if image is too small
                img = img.resize((max(w, self.hr_size), max(h, self.hr_size)), Image.BICUBIC)
                w, h = img.size
            
            left = np.random.randint(0, w - self.hr_size + 1)
            top = np.random.randint(0, h - self.hr_size + 1)
            hr_img = img.crop((left, top, left + self.hr_size, top + self.hr_size))
            
            # Apply augmentation
            hr_img = self.transform(hr_img)
            
            # Create LR via bicubic downsampling
            lr_img = hr_img.resize((self.lr_size, self.lr_size), Image.BICUBIC)
            
            # Convert to tensor and normalize to [-1, 1]
            hr_tensor = self.to_tensor(hr_img) * 2 - 1
            lr_tensor = self.to_tensor(lr_img) * 2 - 1
            
            return lr_tensor, hr_tensor
            
        except Exception:
            # Silent fallback for any other errors
            self.failed_count += 1
            return torch.randn(3, self.lr_size, self.lr_size), torch.randn(3, self.hr_size, self.hr_size)


# ========== AUTOMATIC DATASET SETUP ==========

print("🔍 Checking for BigEarthNet dataset...")

BIGEARTHNET_DIR = None

# Check Google Drive first (fastest if already uploaded)
if os.path.exists('/content/drive/MyDrive/BigEarthNet-S2'):
    BIGEARTHNET_DIR = '/content/drive/MyDrive/BigEarthNet-S2'
    print(f"✓ Found dataset in Google Drive: {BIGEARTHNET_DIR}")

# Check if dataset already in /content/
elif os.path.exists('/content/BigEarthNet-S2') and os.path.isdir('/content/BigEarthNet-S2'):
    BIGEARTHNET_DIR = '/content/BigEarthNet-S2'
    print(f"✓ Found dataset in /content/: {BIGEARTHNET_DIR}")

# Try kagglehub download
else:
    print("\n📥 Dataset not found locally. Attempting download via kagglehub...")
    print("⚠️  This is a large dataset, download may take 30-60 minutes")
    
    try:
        # Install kagglehub
        !pip install -q kagglehub
        
        print("✓ kagglehub installed successfully!")
        print("\n🔄 Starting dataset download...")
        
        import kagglehub
        
        # Download BigEarthNet dataset
        path = kagglehub.dataset_download("immulu/bigearthnetv2-s2-4")
        
        print(f"\n✓ Dataset downloaded successfully!")
        print(f"  Location: {path}")
        
        BIGEARTHNET_DIR = path
        
    except Exception as e:
        print(f"\n❌ Download failed with error: {e}")
        print("\n" + "="*70)
        print("📋 ALTERNATIVE OPTIONS:")
        print("="*70)
        
        print("\n📁 OPTION 1: Upload to Google Drive")
        print("   Step 1: Download BigEarthNet from https://bigearth.net")
        print("   Step 2: Upload to /content/drive/MyDrive/BigEarthNet-S2/")
        print("   Step 3: Re-run this cell")
        
        print("\n🔧 OPTION 2: Try Kaggle API")
        print("   Step 1: Upload kaggle.json to /content/")
        print("   Step 2: Run: !kaggle datasets download -d immulu/bigearthnetv2-s2-4 -p /content --unzip")
        print("   Step 3: Re-run this cell")
        
        print("\n🧪 OPTION 3: Use Synthetic Data for Testing")
        print("   This allows testing the pipeline without real data.")
        print("   Modify this cell and set: BIGEARTHNET_DIR = None")
        print("   Then comment out the raise statement below.")
        
        print("="*70)
        raise FileNotFoundError(f"Dataset download failed. Please try alternative options above.")

# Final validation
if BIGEARTHNET_DIR is None or not os.path.exists(BIGEARTHNET_DIR):
    print("\n" + "="*70)
    print("⚠️  Dataset not found!")
    print("="*70)
    raise FileNotFoundError("Dataset not available. Please use one of the alternative options.")

print(f"\n📡 Loading BigEarthNet-S2 dataset from: {BIGEARTHNET_DIR}")
print("   Using RGB bands: B04 (Red), B03 (Green), B02 (Blue)")
print("   Note: Some patches may fail due to rasterio/numpy compatibility - this is expected")

train_dataset = BigEarthNetDataset(BIGEARTHNET_DIR, hr_size=wandb.config.hr_size, lr_size=wandb.config.lr_size)
val_dataset = BigEarthNetDataset(BIGEARTHNET_DIR, hr_size=wandb.config.hr_size, lr_size=wandb.config.lr_size)

# Reduced num_workers for Colab (avoid multiprocessing issues)
train_loader = DataLoader(train_dataset, batch_size=wandb.config.batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

print(f"\n✓ Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"  (Failed patches will use synthetic data as fallback)")


📡 Loading BigEarthNet-S2 dataset (multi-band satellite imagery)...
   Using RGB bands: B04 (Red), B03 (Green), B02 (Blue)
Found 28937 image patches in /kaggle/input/bigearthnetv2-s2-4/BigEarthNet-S2
Found 28937 image patches in /kaggle/input/bigearthnetv2-s2-4/BigEarthNet-S2
Found 28937 image patches in /kaggle/input/bigearthnetv2-s2-4/BigEarthNet-S2
✓ Train batches: 1809, Val batches: 7235
Found 28937 image patches in /kaggle/input/bigearthnetv2-s2-4/BigEarthNet-S2
✓ Train batches: 1809, Val batches: 7235


In [ ]:
# Cell 5: Training Loop (Stage 1: PSNR + Stage 2: GAN)

def train_stage1(generator, train_loader, val_loader, epochs=25, lr=2e-4):
    """Stage 1: PSNR-oriented training with L1 loss"""
    print("\n" + "="*50)
    print("STAGE 1: PSNR-ORIENTED TRAINING")
    print("="*50)
    
    optimizer = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.9, 0.99))
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    
    generator.train()
    
    for epoch in range(epochs):
        epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for lr_img, hr_img in pbar:
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            optimizer.zero_grad()
            sr_img = generator(lr_img)
            loss = F.l1_loss(sr_img, hr_img)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            pbar.set_postfix({'L1 Loss': f"{loss.item():.4f}"})
        
        avg_loss = epoch_loss / len(train_loader)
        scheduler.step()
        
        # Validation
        generator.eval()
        val_psnr = 0
        with torch.no_grad():
            for lr_img, hr_img in val_loader:
                lr_img = lr_img.to(device)
                hr_img = hr_img.to(device)
                sr_img = generator(lr_img)
                mse = F.mse_loss(sr_img, hr_img)
                psnr = 10 * torch.log10(4 / mse)  # Range [-1,1] → max=2, so 4
                val_psnr += psnr.item()
        val_psnr /= len(val_loader)
        generator.train()
        
        wandb.log({
            'stage1/epoch': epoch + 1,
            'stage1/train_loss': avg_loss,
            'stage1/val_psnr': val_psnr,
            'stage1/lr': optimizer.param_groups[0]['lr']
        })
        
        print(f"Epoch {epoch+1}: Train Loss={avg_loss:.4f}, Val PSNR={val_psnr:.2f} dB")
        
        # Clear cache to prevent OOM on Colab
        torch.cuda.empty_cache()
    
    # Save Stage 1 checkpoint to Google Drive
    save_path = '/content/drive/MyDrive/RFB-ESRGAN-Output/generator_stage1.pth'
    model_state = generator.module.state_dict() if isinstance(generator, nn.DataParallel) else generator.state_dict()
    torch.save(model_state, save_path)
    print(f"✓ Stage 1 complete. Model saved to {save_path}")


def train_stage2(generator, discriminator, train_loader, val_loader, iterations=200000, lr=1e-4):
    """Stage 2: GAN training with perceptual losses, discriminator warmup, and gradient clipping"""
    print("\n" + "="*50)
    print("STAGE 2: GAN TRAINING (WITH DISCRIMINATOR FIXES)")
    print("="*50)
    
    optimizer_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.9, 0.99))
    optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.9, 0.99))
    
    # LR decay schedule - adjusted milestones for 200k iterations
    milestones = [50000, 100000, 150000, 180000]
    scheduler_g = torch.optim.lr_scheduler.MultiStepLR(optimizer_g, milestones=milestones, gamma=0.5)
    scheduler_d = torch.optim.lr_scheduler.MultiStepLR(optimizer_d, milestones=milestones, gamma=0.5)
    
    vgg_loss_fn = VGGPerceptualLoss().to(device)
    gan_loss_fn = GANLoss()
    
    generator.train()
    discriminator.train()
    
    saved_models = []  # Track top-10 models for ensemble
    iter_count = 0
    warmup_iters = wandb.config.stage2_warmup_iters
    d_updates_per_g = wandb.config.d_updates_per_g
    grad_clip_val = wandb.config.grad_clip
    
    print(f"\n🔥 Discriminator Warmup Phase: {warmup_iters} iterations")
    print(f"   Training discriminator alone to catch up with generator...")
    
    # ========== DISCRIMINATOR WARMUP PHASE ==========
    warmup_count = 0
    warmup_loader = iter(train_loader)
    while warmup_count < warmup_iters:
        try:
            lr_img, hr_img, _ = next(warmup_loader)
        except StopIteration:
            warmup_loader = iter(train_loader)
            lr_img, hr_img, _ = next(warmup_loader)
        
        lr_img = lr_img.to(device)
        hr_img = hr_img.to(device)
        
        # Train discriminator only
        optimizer_d.zero_grad()
        with torch.no_grad():
            sr_img = generator(lr_img)
        d_real = discriminator(hr_img)
        d_fake = discriminator(sr_img)
        loss_d = gan_loss_fn(d_real, d_fake, is_disc=True)
        loss_d.backward()
        
        # Gradient clipping for discriminator
        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), grad_clip_val)
        optimizer_d.step()
        
        warmup_count += 1
        if warmup_count % 500 == 0:
            print(f"  Warmup {warmup_count}/{warmup_iters}: D_loss={loss_d.item():.4f}")
    
    print(f"✓ Discriminator warmup complete!\n")
    
    # ========== MAIN GAN TRAINING ==========
    while iter_count < iterations:
        for batch_data in train_loader:
            if iter_count >= iterations:
                break
            
            # Handle both 2-tuple and 3-tuple batch data
            if len(batch_data) == 3:
                lr_img, hr_img, _ = batch_data
            else:
                lr_img, hr_img = batch_data
                
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            # ========== Train Discriminator Multiple Times ==========
            for d_step in range(d_updates_per_g):
                optimizer_d.zero_grad()
                sr_img = generator(lr_img).detach()
                d_real = discriminator(hr_img)
                d_fake = discriminator(sr_img)
                loss_d = gan_loss_fn(d_real, d_fake, is_disc=True)
                loss_d.backward()
                
                # Gradient clipping for discriminator
                torch.nn.utils.clip_grad_norm_(discriminator.parameters(), grad_clip_val)
                optimizer_d.step()
            
            # ========== Train Generator Once ==========
            optimizer_g.zero_grad()
            sr_img = generator(lr_img)
            d_real = discriminator(hr_img).detach()
            d_fake = discriminator(sr_img)
            
            # Total generator loss: L_G = L_pix + λ_vgg*L_vgg + λ_adv*L_adv
            l_pix = F.l1_loss(sr_img, hr_img)
            l_vgg = vgg_loss_fn(sr_img, hr_img)
            l_adv = gan_loss_fn(d_real, d_fake, is_disc=False)
            
            loss_g = (
                wandb.config.lambda_pix * l_pix +
                wandb.config.lambda_vgg * l_vgg +
                wandb.config.lambda_adv * l_adv
            )
            loss_g.backward()
            
            # Gradient clipping for generator
            torch.nn.utils.clip_grad_norm_(generator.parameters(), grad_clip_val)
            optimizer_g.step()
            
            scheduler_g.step()
            scheduler_d.step()
            iter_count += 1
            
            # Logging
            if iter_count % 100 == 0:
                wandb.log({
                    'stage2/iteration': iter_count,
                    'stage2/loss_g': loss_g.item(),
                    'stage2/loss_d': loss_d.item(),
                    'stage2/l_pix': l_pix.item(),
                    'stage2/l_vgg': l_vgg.item(),
                    'stage2/l_adv': l_adv.item(),
                })
            
            # Print progress every 500 iterations
            if iter_count % 500 == 0:
                print(f"Iter {iter_count}/{iterations}: G={loss_g.item():.4f}, D={loss_d.item():.4f}, Pix={l_pix.item():.4f}")
            
            # Save model every 10k iterations for ensemble - save to Google Drive
            if iter_count % 10000 == 0:
                model_path = f'/content/drive/MyDrive/RFB-ESRGAN-Output/generator_iter_{iter_count}.pth'
                model_state = generator.module.state_dict() if isinstance(generator, nn.DataParallel) else generator.state_dict()
                torch.save(model_state, model_path)
                saved_models.append(model_path)
                print(f"✓ Saved checkpoint: {model_path}")
                
                # Clear cache to prevent OOM
                torch.cuda.empty_cache()
    
    print(f"\n✓ Stage 2 complete. {len(saved_models)} checkpoints saved for ensemble.")
    return saved_models


def ensemble_models(generator, model_paths, top_k=10):
    """Average parameters of top-k models"""
    print(f"\nCreating ensemble from top-{top_k} models...")
    
    # Select top-k models (simple: use last k models, or evaluate each)
    selected_models = model_paths[-top_k:] if len(model_paths) >= top_k else model_paths
    
    # Average state dicts
    ensemble_state = OrderedDict()
    for path in selected_models:
        state = torch.load(path, map_location=device)
        for key in state:
            if key not in ensemble_state:
                ensemble_state[key] = state[key].clone()
            else:
                ensemble_state[key] += state[key]
    
    for key in ensemble_state:
        ensemble_state[key] /= len(selected_models)
    
    # Load into generator
    if isinstance(generator, nn.DataParallel):

        generator.module.load_state_dict(ensemble_state)wandb.log({'total_training_hours': total_time})

    else:

        generator.load_state_dict(ensemble_state)print(f"\n📁 All models saved to: /content/drive/MyDrive/RFB-ESRGAN-Output/")

    print(f"{'='*50}")

    save_path = '/content/drive/MyDrive/RFB-ESRGAN-Output/generator_ensemble.pth'print(f"✓ Training complete! Total time: {total_time:.2f} hours")

    torch.save(ensemble_state, save_path)print(f"\n{'='*50}")

    print(f"✓ Ensemble model saved to {save_path}")total_time = (time.time() - start_time) / 3600



ensemble_models(generator, saved_models, top_k=wandb.config.ensemble_models)

# Initialize models# Ensemble top models

generator = Generator(

    num_rrdb=wandb.config.num_rrdb,                            lr=wandb.config.stage2_lr)

    num_rrfdb=wandb.config.num_rrfdb                            iterations=wandb.config.stage2_iters,

).to(device)saved_models = train_stage2(generator, discriminator, train_loader, val_loader,

# Stage 2: GAN training

discriminator = Discriminator().to(device)

             lr=wandb.config.stage1_lr)

# DataParallel is typically not needed on Colab (single GPU)             epochs=wandb.config.stage1_epochs, 

# But keep it for compatibility if using multi-GPU instancestrain_stage1(generator, train_loader, val_loader, 

if torch.cuda.device_count() > 1:# Stage 1: PSNR training

    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel!")

    generator = nn.DataParallel(generator)start_time = time.time()

    discriminator = nn.DataParallel(discriminator)# Execute training

else:

    print("Using single GPU (Colab standard)")print(f"Discriminator params: {sum(p.numel() for p in discriminator.parameters())/1e6:.2f}M")

print(f"Generator params: {sum(p.numel() for p in generator.parameters())/1e6:.2f}M")

In [ ]:
# Cell 6: Comprehensive Evaluation Metrics

# Install additional required packages for advanced metrics
!pip install -q lpips pytorch-msssim scikit-learn seaborn

import lpips
from pytorch_msssim import ssim, ms_ssim
from sklearn.metrics import confusion_matrix, cohen_kappa_score, precision_recall_fscore_support, top_k_accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict

# ========== 1. SUPER-RESOLUTION METRICS ==========

class SuperResolutionMetrics:
    """Comprehensive SR evaluation metrics"""
    def __init__(self, device):
        self.device = device
        # LPIPS loss network (Alex)
        self.lpips_fn = lpips.LPIPS(net='alex').to(device)
        
    def calculate_psnr(self, sr, hr):
        """Peak Signal-to-Noise Ratio"""
        mse = F.mse_loss(sr, hr)
        psnr = 10 * torch.log10(4 / mse)  # Range [-1,1] → max=2, so 4
        return psnr.item()
    
    def calculate_ssim(self, sr, hr):
        """Structural Similarity Index"""
        # Normalize from [-1,1] to [0,1]
        sr_norm = (sr + 1) / 2
        hr_norm = (hr + 1) / 2
        ssim_val = ssim(sr_norm, hr_norm, data_range=1.0, size_average=True)
        return ssim_val.item()
    
    def calculate_lpips(self, sr, hr):
        """Learned Perceptual Image Patch Similarity"""
        lpips_val = self.lpips_fn(sr, hr)
        return lpips_val.mean().item()
    
    def calculate_ndvi_error(self, sr, hr):
        """Spectral Consistency - NDVI Error for vegetation index accuracy"""
        # NDVI = (NIR - Red) / (NIR + Red)
        # For RGB images, approximate using Red channel (B04 equivalent)
        # This is a simplified version - full NDVI needs NIR band
        
        # Extract red channel (assuming channel 0 is red after normalization)
        sr_red = sr[:, 0:1, :, :]  # Red channel
        hr_red = hr[:, 0:1, :, :]
        
        # Simplified NDVI approximation (for demonstration)
        # In practice, you'd need actual NIR and Red bands
        ndvi_error = F.l1_loss(sr_red, hr_red)
        return ndvi_error.item()
    
    def evaluate_batch(self, sr, hr):
        """Evaluate all SR metrics on a batch"""
        metrics = {
            'psnr': self.calculate_psnr(sr, hr),
            'ssim': self.calculate_ssim(sr, hr),
            'lpips': self.calculate_lpips(sr, hr),
            'ndvi_error': self.calculate_ndvi_error(sr, hr)
        }
        return metrics


# ========== 2. BASELINE COMPARISON MODELS ==========

class BicubicUpsampler:
    """Baseline bicubic interpolation"""
    def __init__(self, scale_factor=8):
        self.scale_factor = scale_factor
    
    def __call__(self, lr_img):
        return F.interpolate(lr_img, scale_factor=self.scale_factor, mode='bicubic', align_corners=False)


class SimpleSRCNN(nn.Module):
    """Lightweight SRCNN baseline for comparison"""
    def __init__(self, scale_factor=8):
        super(SimpleSRCNN, self).__init__()
        self.scale_factor = scale_factor
        
        # SRCNN: 3 conv layers
        self.conv1 = nn.Conv2d(3, 64, 9, padding=4)
        self.conv2 = nn.Conv2d(64, 32, 1, padding=0)
        self.conv3 = nn.Conv2d(32, 3, 5, padding=2)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        # Bicubic upsampling first
        x = F.interpolate(x, scale_factor=self.scale_factor, mode='bicubic', align_corners=False)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)
        return torch.tanh(x)


# ========== 3. COMPARATIVE EVALUATION ==========

def comparative_evaluation(generator, val_loader, device, num_samples=50):
    """Compare RFB-ESRGAN against baselines"""
    print("\n" + "="*70)
    print("COMPARATIVE EVALUATION: RFB-ESRGAN vs. Baselines")
    print("="*70)
    
    # Initialize models
    sr_metrics = SuperResolutionMetrics(device)
    bicubic = BicubicUpsampler(scale_factor=8)
    srcnn = SimpleSRCNN(scale_factor=8).to(device)
    srcnn.eval()
    
    # Results storage
    results = {
        'ours': defaultdict(list),
        'bicubic': defaultdict(list),
        'srcnn': defaultdict(list)
    }
    
    inference_times = {
        'ours': [],
        'bicubic': [],
        'srcnn': []
    }
    
    generator.eval()
    sample_count = 0
    
    print(f"\n📊 Evaluating on {num_samples} samples...")
    
    with torch.no_grad():
        for lr_img, hr_img in tqdm(val_loader, desc="Evaluating"):
            if sample_count >= num_samples:
                break
            
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            # ========== RFB-ESRGAN (Ours) ==========
            start_time = time.time()
            sr_ours = generator(lr_img)
            inference_times['ours'].append(time.time() - start_time)
            
            metrics_ours = sr_metrics.evaluate_batch(sr_ours, hr_img)
            for k, v in metrics_ours.items():
                results['ours'][k].append(v)
            
            # ========== Bicubic Baseline ==========
            start_time = time.time()
            sr_bicubic = bicubic(lr_img)
            inference_times['bicubic'].append(time.time() - start_time)
            
            metrics_bicubic = sr_metrics.evaluate_batch(sr_bicubic, hr_img)
            for k, v in metrics_bicubic.items():
                results['bicubic'][k].append(v)
            
            # ========== SRCNN Baseline ==========
            start_time = time.time()
            sr_srcnn = srcnn(lr_img)
            inference_times['srcnn'].append(time.time() - start_time)
            
            metrics_srcnn = sr_metrics.evaluate_batch(sr_srcnn, hr_img)
            for k, v in metrics_srcnn.items():
                results['srcnn'][k].append(v)
            
            sample_count += lr_img.size(0)
    
    # ========== Calculate Average Metrics ==========
    print("\n" + "="*70)
    print("RESULTS SUMMARY")
    print("="*70)
    
    comparison_table = []
    
    for model_name in ['bicubic', 'srcnn', 'ours']:
        avg_metrics = {k: np.mean(v) for k, v in results[model_name].items()}
        avg_time = np.mean(inference_times[model_name]) * 1000  # ms
        
        print(f"\n{model_name.upper()}:")
        print(f"  PSNR: {avg_metrics['psnr']:.2f} dB")
        print(f"  SSIM: {avg_metrics['ssim']:.4f}")
        print(f"  LPIPS: {avg_metrics['lpips']:.4f} (lower is better)")
        print(f"  NDVI Error: {avg_metrics['ndvi_error']:.4f}")
        print(f"  Inference Time: {avg_time:.2f} ms/image")
        
        comparison_table.append({
            'model': model_name,
            **avg_metrics,
            'inference_time_ms': avg_time
        })
    
    # ========== Calculate Improvement Deltas ==========
    print("\n" + "="*70)
    print("IMPROVEMENT vs. BASELINES")
    print("="*70)
    
    ours_psnr = np.mean(results['ours']['psnr'])
    ours_ssim = np.mean(results['ours']['ssim'])
    bicubic_psnr = np.mean(results['bicubic']['psnr'])
    bicubic_ssim = np.mean(results['bicubic']['ssim'])
    srcnn_psnr = np.mean(results['srcnn']['psnr'])
    srcnn_ssim = np.mean(results['srcnn']['ssim'])
    
    delta_psnr_bicubic = ours_psnr - bicubic_psnr
    delta_ssim_bicubic = ours_ssim - bicubic_ssim
    delta_psnr_srcnn = ours_psnr - srcnn_psnr
    delta_ssim_srcnn = ours_ssim - srcnn_ssim
    
    print(f"\nΔPSNR vs. Bicubic: +{delta_psnr_bicubic:.2f} dB")
    print(f"ΔSSIM vs. Bicubic: +{delta_ssim_bicubic:.4f}")
    print(f"ΔPSNR vs. SRCNN: +{delta_psnr_srcnn:.2f} dB")
    print(f"ΔSSIM vs. SRCNN: +{delta_ssim_srcnn:.4f}")
    
    # ========== Model Efficiency ==========
    print("\n" + "="*70)
    print("MODEL EFFICIENCY METRICS")
    print("="*70)
    
    # Parameter count
    def count_parameters(model):
        if isinstance(model, nn.DataParallel):
            return sum(p.numel() for p in model.module.parameters())
        return sum(p.numel() for p in model.parameters())
    
    ours_params = count_parameters(generator)
    srcnn_params = count_parameters(srcnn)
    
    print(f"\nParameter Count:")
    print(f"  RFB-ESRGAN (Ours): {ours_params/1e6:.2f}M parameters")
    print(f"  SRCNN: {srcnn_params/1e6:.2f}M parameters")
    print(f"  Bicubic: 0M parameters (no learning)")
    
    # Parameter efficiency: PSNR gain per million parameters
    psnr_per_param_ours = (ours_psnr - bicubic_psnr) / (ours_params / 1e6)
    psnr_per_param_srcnn = (srcnn_psnr - bicubic_psnr) / (srcnn_params / 1e6)
    
    print(f"\nParameter Efficiency (ΔPSNR per 1M params):")
    print(f"  RFB-ESRGAN: {psnr_per_param_ours:.3f} dB/M")
    print(f"  SRCNN: {psnr_per_param_srcnn:.3f} dB/M")
    
    # ========== System Performance Metrics ==========
    print("\n" + "="*70)
    print("SYSTEM PERFORMANCE METRICS")
    print("="*70)
    
    # Throughput (FPS)
    avg_time_ours = np.mean(inference_times['ours'])
    fps_ours = 1.0 / avg_time_ours if avg_time_ours > 0 else 0
    
    print(f"\nInference Latency (Ours): {avg_time_ours*1000:.2f} ms")
    print(f"Throughput (Ours): {fps_ours:.2f} FPS")
    
    # GPU Memory footprint
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            _ = generator(lr_img)
        memory_allocated = torch.cuda.max_memory_allocated() / 1024**2  # MB
        print(f"GPU Memory Footprint: {memory_allocated:.2f} MB")
    
    # ========== Log to WandB ==========
    wandb.log({
        'eval/psnr_ours': ours_psnr,
        'eval/ssim_ours': ours_ssim,
        'eval/lpips_ours': np.mean(results['ours']['lpips']),
        'eval/psnr_bicubic': bicubic_psnr,
        'eval/ssim_bicubic': bicubic_ssim,
        'eval/psnr_srcnn': srcnn_psnr,
        'eval/ssim_srcnn': srcnn_ssim,
        'eval/delta_psnr_vs_bicubic': delta_psnr_bicubic,
        'eval/delta_ssim_vs_bicubic': delta_ssim_bicubic,
        'eval/delta_psnr_vs_srcnn': delta_psnr_srcnn,
        'eval/delta_ssim_vs_srcnn': delta_ssim_srcnn,
        'eval/inference_time_ms': avg_time_ours * 1000,
        'eval/throughput_fps': fps_ours,
        'eval/parameters_millions': ours_params / 1e6,
        'eval/psnr_per_param': psnr_per_param_ours,
    })
    
    # ========== Visualize Comparison ==========
    plt.figure(figsize=(15, 10))
    
    # Plot 1: PSNR Comparison
    plt.subplot(2, 3, 1)
    models = ['Bicubic', 'SRCNN', 'Ours']
    psnr_values = [bicubic_psnr, srcnn_psnr, ours_psnr]
    plt.bar(models, psnr_values, color=['#ff7f0e', '#2ca02c', '#1f77b4'])
    plt.ylabel('PSNR (dB)')
    plt.title('PSNR Comparison')
    plt.grid(axis='y', alpha=0.3)
    
    # Plot 2: SSIM Comparison
    plt.subplot(2, 3, 2)
    ssim_values = [bicubic_ssim, srcnn_ssim, ours_ssim]
    plt.bar(models, ssim_values, color=['#ff7f0e', '#2ca02c', '#1f77b4'])
    plt.ylabel('SSIM')
    plt.title('SSIM Comparison')
    plt.grid(axis='y', alpha=0.3)
    
    # Plot 3: Inference Time
    plt.subplot(2, 3, 3)
    times = [np.mean(inference_times['bicubic'])*1000, 
             np.mean(inference_times['srcnn'])*1000,
             np.mean(inference_times['ours'])*1000]
    plt.bar(models, times, color=['#ff7f0e', '#2ca02c', '#1f77b4'])
    plt.ylabel('Time (ms)')
    plt.title('Inference Time')
    plt.grid(axis='y', alpha=0.3)
    
    # Plot 4: LPIPS (lower is better)
    plt.subplot(2, 3, 4)
    lpips_values = [np.mean(results['bicubic']['lpips']),
                    np.mean(results['srcnn']['lpips']),
                    np.mean(results['ours']['lpips'])]
    plt.bar(models, lpips_values, color=['#ff7f0e', '#2ca02c', '#1f77b4'])
    plt.ylabel('LPIPS (lower is better)')
    plt.title('Perceptual Quality (LPIPS)')
    plt.grid(axis='y', alpha=0.3)
    
    # Plot 5: Parameter Efficiency
    plt.subplot(2, 3, 5)
    efficiency = [0, psnr_per_param_srcnn, psnr_per_param_ours]
    plt.bar(models, efficiency, color=['#ff7f0e', '#2ca02c', '#1f77b4'])
    plt.ylabel('ΔPSNR per 1M Parameters')
    plt.title('Parameter Efficiency')
    plt.grid(axis='y', alpha=0.3)
    
    # Plot 6: Overall Performance Radar
    plt.subplot(2, 3, 6)
    categories = ['PSNR\n(norm)', 'SSIM\n(norm)', 'Speed\n(inv)', 'LPIPS\n(inv)']
    
    # Normalize metrics to [0, 1] for radar plot
    def normalize(values):
        min_val, max_val = min(values), max(values)
        return [(v - min_val) / (max_val - min_val) if max_val > min_val else 0.5 for v in values]
    
    ours_radar = [
        normalize(psnr_values)[2],
        normalize(ssim_values)[2],
        1 - normalize(times)[2],  # Invert (faster is better)
        1 - normalize(lpips_values)[2]  # Invert (lower is better)
    ]
    
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    ours_radar += ours_radar[:1]
    angles += angles[:1]
    
    ax = plt.subplot(2, 3, 6, projection='polar')
    ax.plot(angles, ours_radar, 'o-', linewidth=2, label='RFB-ESRGAN', color='#1f77b4')
    ax.fill(angles, ours_radar, alpha=0.25, color='#1f77b4')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_ylim(0, 1)
    ax.set_title('Overall Performance')
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    ax.grid(True)
    
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/RFB-ESRGAN-Output/comparative_evaluation.png', dpi=150, bbox_inches='tight')
    wandb.log({"comparative_evaluation": wandb.Image(plt)})
    plt.show()
    
    print(f"\n✓ Comparative evaluation complete!")
    print(f"  Visualization saved to: /content/drive/MyDrive/RFB-ESRGAN-Output/comparative_evaluation.png")
    
    return comparison_table


# ========== 4. VISUAL QUALITY SAMPLES ==========

def visualize_quality_comparison(generator, val_loader, device, num_samples=5):
    """Visualize SR results for visual quality assessment"""
    print("\n📸 Generating visual quality samples...")
    
    bicubic = BicubicUpsampler(scale_factor=8)
    generator.eval()
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 4))
    
    with torch.no_grad():
        for i, (lr_img, hr_img) in enumerate(val_loader):
            if i >= num_samples:
                break
            
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            
            # Generate SR images
            sr_ours = generator(lr_img)
            sr_bicubic = bicubic(lr_img)
            
            # Take first image in batch
            lr_np = ((lr_img[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
            hr_np = ((hr_img[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
            sr_ours_np = ((sr_ours[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
            sr_bicubic_np = ((sr_bicubic[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
            
            # Plot
            axes[i, 0].imshow(lr_np)
            axes[i, 0].set_title(f'LR Input ({lr_img.shape[2]}x{lr_img.shape[3]})')
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(sr_bicubic_np)
            axes[i, 1].set_title('Bicubic')
            axes[i, 1].axis('off')
            
            axes[i, 2].imshow(sr_ours_np)
            axes[i, 2].set_title('RFB-ESRGAN (Ours)')
            axes[i, 2].axis('off')
            
            axes[i, 3].imshow(hr_np)
            axes[i, 3].set_title(f'HR Ground Truth ({hr_img.shape[2]}x{hr_img.shape[3]})')
            axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/RFB-ESRGAN-Output/visual_quality_samples.png', dpi=150, bbox_inches='tight')
    wandb.log({"visual_quality_samples": wandb.Image(plt)})
    plt.show()
    
    print(f"✓ Visual samples saved to: /content/drive/MyDrive/RFB-ESRGAN-Output/visual_quality_samples.png")


# ========== EXECUTE COMPREHENSIVE EVALUATION ==========

print("\n" + "="*70)
print("🚀 STARTING COMPREHENSIVE EVALUATION")
print("="*70)

# Load best model (ensemble)
ensemble_path = '/content/drive/MyDrive/RFB-ESRGAN-Output/generator_ensemble.pth'
ensemble_state = torch.load(ensemble_path)
if isinstance(generator, nn.DataParallel):
    generator.module.load_state_dict(ensemble_state)
else:
    generator.load_state_dict(ensemble_state)

# Run comprehensive evaluation
comparison_results = comparative_evaluation(generator, val_loader, device, num_samples=100)

# Generate visual quality samples
visualize_quality_comparison(generator, val_loader, device, num_samples=5)

# ========== FINAL SUMMARY ==========
print("\n" + "="*70)
print("✅ COMPREHENSIVE EVALUATION COMPLETE")
print("="*70)
print("\n📊 Metrics Evaluated:")
print("  ✓ Super-Resolution: PSNR, SSIM, LPIPS, NDVI Error")
print("  ✓ Comparative: Δ vs. Bicubic & SRCNN")
print("  ✓ Efficiency: Parameters, Speed, Memory")
print("  ✓ System Performance: Latency, Throughput, GPU Memory")
print("\n📁 Results saved to:")
print("  • /content/drive/MyDrive/RFB-ESRGAN-Output/comparative_evaluation.png")
print("  • /content/drive/MyDrive/RFB-ESRGAN-Output/visual_quality_samples.png")
print(f"\n🌐 WandB Dashboard: {wandb.run.url}")

wandb.finish()
